____________________
# Exploring TEI with E Tree

Here we will explore XML files encoded in the Text Encoding Initiative (TEI) format. TEI is a widely used standard for representing texts in digital form, particularly in the humanities. TEI files are XML files that contain rich metadata and structural information about texts, making them suitable for various types of analysis.



* **access**: downloading, storing, reading, processing XML files
* **analysis**: performing basic quantitative and qualitative analysis of XML files
* **interpretation**: exploring the meaning and utility of XML files from both analytical and creative perspectives 

As we deal with these files, we will make use of lxml.etree for XML parsing.  


XML files are guided by **markup rules**, which you can read more about [here](https://www.w3schools.com/xml/xml_syntax.asp) and consist of **elements**, which you can dive into [here](https://www.w3schools.com/xml/xml_elements.asp).

TEI files are primarily XML files with specific sections, elements, and structure. You can learn more about TEI [here](https://tei-c.org/).


----

## 1. Import Libraries

In [12]:
import requests
import textwrap
import base64
import pandas as pd
from lxml import etree
import networkx as nx
from pyvis.network import Network
from IPython.display import HTML, display
import pronouncing

import plotly.io as pio
pio.renderers.default = "plotly_mimetype+notebook"


## 2. Import TEI File



In [13]:


url = 'https://ebba.english.ucsb.edu/ballad/37021/ebba-xml-37021'

# Load the XML from the web
response = requests.get(url)
xml_content = response.text

# Parse with lxml, using recover=True to handle entity issues
parser = etree.XMLParser(recover=True)
root = etree.fromstring(xml_content.encode('utf-8'), parser=parser)



## 3. Exploring the Header

#### Summary: The Four Sections of `<teiHeader>`

| Section | Tag | Main Child Elements | Purpose |
|---|---|---|---|
| File Description | `<fileDesc>` | `<titleStmt>`, `<editionStmt>`, `<publicationStmt>`, `<notesStmt>`, `<sourceDesc>` | Who made the file, who published it, and where the original source came from |
| Encoding Description | `<encodingDesc>` | `<editorialDecl>`, `<classDecl>` → `<taxonomy>` → `<category>` | How the document was transcribed and how it is classified |
| Profile Description | `<profileDesc>` | `<creation>`, `<textClass>` → `<keywords>` → `<item>` | When and where the file was created; which subject keywords apply |
| Revision Description | `<revisionDesc>` | `<change>` → `<date>`, `<respStmt>`, `<item>` | Chronological log of every editorial change — who, when, and what |


In [14]:
# fileDesc

header = root.find('teiHeader')
fd = header.find('fileDesc')

print("=== fileDesc ===")

title_stmt = fd.find('titleStmt')
print("\n-- titleStmt --")
for title in title_stmt.findall('title'):
    print(f"  title:   {''.join(title.itertext()).strip()[:80]}")
for author in title_stmt.findall('author'):
    print(f"  author:  {''.join(author.itertext()).strip() or '(anonymous)'}")
for sponsor in title_stmt.findall('sponsor'):
    print(f"  sponsor: {''.join(sponsor.itertext()).strip()}")
for rs in title_stmt.findall('respStmt'):
    resp = ''.join(rs.find('resp').itertext()).strip() if rs.find('resp') is not None else ''
    name = ''.join(rs.find('name').itertext()).strip() if rs.find('name') is not None else ''
    print(f"  respStmt: {resp} — {name}")

pub = fd.find('publicationStmt')
print("\n-- publicationStmt --")
for tag in ['publisher', 'pubPlace', 'date']:
    el = pub.find(tag)
    if el is not None:
        print(f"  {tag}: {''.join(el.itertext()).strip()}")
for idno in pub.findall('idno'):
    print(f"  idno: {''.join(idno.itertext()).strip()}")

notes = fd.find('notesStmt')
print(f"\n-- notesStmt -- ({len(notes.findall('note'))} notes)")
for note in notes.findall('note'):
    print(f"  note: {''.join(note.itertext()).strip()[:80]}")

src = fd.find('sourceDesc')
bibl = src.find('.//bibl')
print("\n-- sourceDesc / bibl --")
for tag in ['title', 'author', 'biblScope']:
    for el in bibl.findall(tag):
        print(f"  {tag}: {''.join(el.itertext()).strip()[:80]}")
imprint = bibl.find('imprint')
if imprint is not None:
    date = imprint.find('date')
    if date is not None:
        print(f"  imprint date: {''.join(date.itertext()).strip()}")


=== fileDesc ===

-- titleStmt --
  title:   THE / True Lovers Knot Untied: / Being the right PATH whereby to advise Princely
  author:  (anonymous)
  sponsor: University of California - Santa Barbara
  sponsor: The Early Modern Center
  sponsor: English Broadside Ballad Archive (EBBA)
  respStmt: Director — Patricia Fumerton
  respStmt: Associate Director — Carl G Stahmer

-- publicationStmt --
  publisher: Early Modern Center, University of California Santa Barbara
  pubPlace: Santa Barbara, CA
  date: 05/20/2021
  idno: 37021
  idno: R227409

-- notesStmt -- (5 notes)
  note: 1
  note: Frog's Ga[l]liard, &c.
  note: Frog Galliard, The
  note: Frog's Galliard, etc.
  note: AS I to Ireland did pass, / I saw a Ship at Anchor lay,

-- sourceDesc / bibl --
  title: THE / True Lovers Knot Untied: / Being the right PATH whereby to advise Princely
  title: THE True Lover's Knot Untied: Being the right PATH whereby to advise Princely Vi
  author: 
  biblScope: 1: 188
  imprint date: ?-?


In [15]:
# encodingDesc
ed = header.find('encodingDesc')

print("=== encodingDesc ===")

ed_decl = ed.find('editorialDecl')
paras = ed_decl.findall('p')
print(f"\n-- editorialDecl -- ({len(paras)} policy paragraphs)")
for i, p in enumerate(paras, 1):
    print(f"  [{i}] {''.join(p.itertext()).strip()[:100]}")

print("\n-- classDecl / taxonomies --")
for taxonomy in ed.findall('.//taxonomy'):
    tax_id = taxonomy.get('id', '(no id)')
    categories = taxonomy.findall('category')
    print(f"\n  taxonomy id={tax_id}  ({len(categories)} categories)")
    for cat in categories[:8]:
        desc = cat.find('catDesc')
        if desc is not None:
            print(f"    {cat.get('id','')}: {''.join(desc.itertext()).strip()}")
    if len(categories) > 8:
        print(f"    … and {len(categories)-8} more")


=== encodingDesc ===

-- editorialDecl -- (7 policy paragraphs)
  [1] This document follows the guidelines specified for TEI.
  [2] XML Generated Automatically  at 5/20/2021 3:40:51 AM Using EMC
  [3] XBallad Parsing Engine developed by Carl G Stahmer.
  [4] TEI Template developed by Gerald Egan and Modified by Carl Stahmer
  [5] All apostrophes are encoded as &apos;.
  [6] Any dashs occurring in line breaks have been removed;
  [7] All dashs are encoded as &dash; and all em dashes as &mdash;.

-- classDecl / taxonomies --

  taxonomy id=EMCKEYWORDS  (53 categories)
    emc.7: advice
    emc.23: affliction / health
    emc.15: alcohol
    emc.52: Americas
    emc.21: animals / nature
    emc.47: Bible / biblical figures
    emc.53: buildings / architecture
    emc.28: catastrophe
    … and 45 more

  taxonomy id=LOCSH  (0 categories)


In [16]:

# profileDesc
profile_desc = header.find('profileDesc')

print("=== profileDesc ===")

creation = profile_desc.find('creation')
if creation is not None:
    date = creation.find('date')
    name = creation.find('name')
    print("\n-- creation --")
    if date is not None:
        print(f"  date: {''.join(date.itertext()).strip()}")
    if name is not None:
        print(f"  place: {''.join(name.itertext()).strip()}")

print("\n-- textClass / keywords --")
for item in profile_desc.findall('.//keywords//item'):
    print(f"  item: {''.join(item.itertext()).strip()}")


=== profileDesc ===

-- creation --
  date: 5/20/2021
  place: Santa Barbara, California, United States of America

-- textClass / keywords --
  item: Ballads, English 17th century
  item: Broadsides, England 17th century


In [17]:
# revisionDesc
rev_desc = header.find('revisionDesc')
changes = rev_desc.findall('change')

print(f"=== revisionDesc ({len(changes)} changes) ===")
for i, change in enumerate(changes, 1):
    date = change.find('date')
    rs   = change.find('respStmt')
    item = change.find('item')
    date_text = ''.join(date.itertext()).strip() if date is not None else ''
    resp_text = ''.join(rs.find('resp').itertext()).strip() if rs is not None and rs.find('resp') is not None else ''
    name_text = ''.join(rs.find('name').itertext()).strip() if rs is not None and rs.find('name') is not None else ''
    item_text = ''.join(item.itertext()).strip()[:80] if item is not None else ''
    print(f"\n  change {i}: {date_text}")
    print(f"    {resp_text} — {name_text}")
    print(f"    {item_text}")


=== revisionDesc (6 changes) ===

  change 1: 5/20/2021 3:40:51 AM
    XBallad — Raychawdhuri, Anita
    Created XML Version of Ballad

  change 2: 5/20/2021 3:40:51 AM
    Transcription Supervisor — McCants, Kristen
    Transcription of ballad manuscript

  change 3: 5/20/2021 3:40:51 AM
    Double-Key Comparison and Merging — McCants, Kristen
    Transcription of ballad manuscript

  change 4: 5/20/2021 3:40:51 AM
    Transcriptionist Two — Pettersson Peeker, Aili
    Transcription of ballad manuscript

  change 5: 5/20/2021 3:40:51 AM
    Transcriptionist One — Raychawdhuri, Anita
    Transcription of ballad manuscript

  change 6: 3/19/2019
    Bibliographer — Kristen McCants
    Initial Ballad Catalogue Record Created


## 3. Visualizing TEI Structure as a Network

One of the best ways to understand any XML document is to visualize its **tree structure** as an interactive network. In the graphs below:

- Each **node** is a TEI element (a tag like `<teiHeader>` or `<l>`)
- Each **directed edge** points from a parent element to its child
- **Node size** reflects how many descendant elements it contains — larger nodes are structurally richer
- **Node color** encodes tree depth — the root is one color, its children another, and so on

Because TEI files are strictly hierarchical (every element has exactly one parent), these networks are **trees**, not webs. The visual layout makes it easy to see which elements are "hubs" of content and which are leaf nodes.

Use the interactive controls to:
- **Drag** nodes to untangle the layout
- **Scroll** to zoom in and read labels
- **Hover** over a node to highlight its immediate connections

In [18]:
def _local_tag(element):
    tag = element.tag
    return tag.split("}", 1)[1] if "}" in tag else tag


def format_element_et(element, wrap_length=20, exclude=[]):
    attrs_list = []
    for a, v in element.attrib.items():
        if a in exclude:
            continue
        attrs_list.append(f"{a}={v}")
    tag_name = _local_tag(element)
    attrs_str = " ".join(attrs_list)
    formatted_string = f"{tag_name} ({attrs_str})" if attrs_list else tag_name
    return textwrap.fill(formatted_string, wrap_length)


def create_network_et(element, with_attributes=False, attrs_to_exclude=[], max_depth=None):
    all_elements = list(element.iter())
    elem_to_idx = {el: i for i, el in enumerate(all_elements)}

    depth_map = {}
    for el in all_elements:
        parent = el.getparent()
        if parent is None or parent not in depth_map:
            depth_map[el] = 0
        else:
            depth_map[el] = depth_map[parent] + 1

    if max_depth is not None:
        all_elements = [el for el in all_elements if depth_map[el] <= max_depth]

    G = nx.DiGraph()

    for node in all_elements:
        depth = depth_map.get(node, 0)
        tag_name = _local_tag(node)
        G.add_node(
            elem_to_idx[node],
            label=format_element_et(node, exclude=attrs_to_exclude) if with_attributes else tag_name,
            value=sum(1 for _ in node.iter()) - 1,
            group=depth,
            level=depth,
            scaling={"label": {"enabled": True}},
        )

    for node in all_elements:
        parent_idx = elem_to_idx[node]
        for child in node:
            child_idx = elem_to_idx.get(child)
            if child_idx is not None and (max_depth is None or depth_map.get(child, 0) <= max_depth):
                G.add_edge(
                    parent_idx, child_idx,
                    arrows="to",
                    id=f"{parent_idx}_{_local_tag(node)}|{child_idx}_{_local_tag(child)}",
                )

    return G


def display_network(network,
                    filename="tmp.html",
                    width="100%",
                    height="650px",
                    bgcolor="white",
                    font_color="black"):
    # cdn_resources="in_line" bundles vis.js into the file -- no CDN needed
    nt = Network(width=width, height=height, bgcolor=bgcolor,
                 font_color=font_color)
    nt.from_nx(network)
    nt.save_graph(filename)
    # Embed as a base64 data-URI iframe -- works in VSCode, JupyterLab, and classic Jupyter
    with open(filename, "rb") as f:
        b64 = base64.b64encode(f.read()).decode()
    display(HTML(
        f'<iframe src="data:text/html;base64,{b64}" '
        f'width="{width}" height="{height}" frameborder="0"></iframe>'
    ))


### 3a. Network of the TEI Header

The **`<teiHeader>`** is the metadata section of every TEI document — analogous to a library catalogue record attached to the text itself. It is divided into four major sections:

| Section | Tag | What it contains |
|---|---|---|
| File description | `<fileDesc>` | Title, authors, sponsors, editors, publication info, source bibliography |
| Encoding description | `<encodingDesc>` | Editorial policies and classification schemes (taxonomies) |
| Profile description | `<profileDesc>` | Date and place of creation, subject keywords |
| Revision description | `<revisionDesc>` | Change log — who edited the file and when |

**What to look for in the network:**

- `<fileDesc>` is the largest node because it contains the most nested content: titles, responsibility statements, publication details, and source bibliography all live here.
- `<encodingDesc>` sprouts a huge fan of `<category>` nodes via its `<taxonomy>` — these are the EBBA subject keywords (love, crime, royalty, etc.) used to classify ballads across the archive.
- `<revisionDesc>` repeats a regular pattern: each `<change>` contains `<date>`, `<respStmt>`, and `<item>` — a record of every editorial intervention.

In [19]:
# Overview: teiHeader with its four direct children
header = root.find('teiHeader')
G_overview = create_network_et(header, max_depth=1)
display_network(G_overview, filename='tei_overview.html', height='400px')


/Users/rfreedma/anaconda3/envs/encoding_music/lib/python3.10/site-packages/IPython/core/display.py:431: UserWarning:

Consider using IPython.display.IFrame instead



In [20]:
# Full network for each direct child of teiHeader
for child in header:
    tag = _local_tag(child)
    print(f'── {tag} ──')
    G = create_network_et(child)
    display_network(G, filename=f'tei_{tag}.html', height='600px')


── fileDesc ──


── encodingDesc ──


── profileDesc ──


── revisionDesc ──


### 3bNetwork of the Complete TEI Document

The full document adds the **`<text>`** element alongside `<teiHeader>`, showing the two-part architecture of every TEI file:

```
TEI.2
├── teiHeader   (metadata — who, what, when, where, how it was encoded)
└── text        (the actual content of the document)
```

**Inside `<text>` for this ballad:**

| Element | Role |
|---|---|
| `<body>` | The main content area |
| `<div type="ballad">` | The whole ballad as a division |
| `<div type="part">` | Each column or part of the broadside sheet |
| `<lg type="stanza">` | A line group — one stanza |
| `<l>` | A single line of verse |

**What to look for:**

- The `<text>` branch (left side) is far larger than `<teiHeader>` — 25 stanzas × 4 lines each produce many `<l>` leaf nodes, visible as the dense outer ring of the text cluster.
- The `<div type="part">` nodes are intermediate hubs, each gathering several stanzas.
- Notice how the metadata tree (right side) and the text tree (left side) are structurally very different: metadata is **wide and varied** (many different element names), while the text body is **deep and repetitive** (the same `<lg>` / `<l>` pattern over and over).

> **Think about it:** What does this structural difference tell us about the two jobs TEI asks a document to do — *describing* a text vs. *encoding* it?

In [21]:
G_full = create_network_et(root, with_attributes=True, attrs_to_exclude=['n', 'id'])
display_network(G_full, filename="tei_full_network.html", height="800px")

##  4. Explore the Body:  Line Groups and Lines

- Here we will explore the body of the ballad, which is organized into **line groups** (`<lg>`) and **lines** (`<l>`). Each `<lg>` represents a stanza, and each `<l>` represents a single line of verse.
- We will extract the text of each line, identify the rhyme words, and analyze the rhyme scheme of the ballad.

## 4a. Extracting Lines and Rhyme Words

In [22]:
rows = []
lg_num = 0
for elem in root.iter():
    if _local_tag(elem) == 'lg':
        lg_num += 1
        line_num = 0
        for child in elem:
            if _local_tag(child) == 'l':
                line_num += 1
                text = ''.join(child.itertext()).strip()
                rows.append({'lg': lg_num, 'line': line_num, 'text': text})

df = pd.DataFrame(rows)
df['rhyme_word'] = df['text'].str.split().str[-1].str.strip('.,;:!?"\'-')
df


,lg,line,text,rhyme_word
0,1,1,"AS I to Ireland did pass,",pass
1,1,2,"I saw a Ship at Anchor lay,",lay
2,1,3,"Another Ship likewise there was,",was
3,1,4,which from fair England took her way.,way
4,2,1,"This Ship that sa[i]l'd from fair England,",England
...,...,...,...,...
95,24,4,but now am forc'd to part with thee.,thee
96,25,1,"At this sad Meeting she had cause,",cause
97,25,2,"in heart and mind to grieve full fare,",fare
98,25,3,"After that time Arabella fair,",fair


### 4b. Analyze Rhymes with the `pronouncing` Library

This library works phonetically, so it can identify rhymes even when the spelling is different (e.g., "love" and "dove"). It uses the CMU Pronouncing Dictionary, which is a widely used resource in computational linguistics for English pronunciation.

In [23]:


def assign_rhyme_scheme(group):
    rhyme_map = {}
    scheme = []
    counter = 0
    for word in group['rhyme_word'].str.lower():
        key = word[-2:]
        if key not in rhyme_map:
            rhyme_map[key] = chr(ord('A') + counter)
            counter += 1
        scheme.append(rhyme_map[key])
    return pd.Series(scheme, index=group.index)

df['rhyme_scheme'] = df.groupby('lg', group_keys=False,).apply(assign_rhyme_scheme, include_groups=False)
df[['lg', 'line', 'rhyme_word', 'rhyme_scheme']]


,lg,line,rhyme_word,rhyme_scheme
0,1,1,pass,A
1,1,2,lay,B
2,1,3,was,C
3,1,4,way,B
4,2,1,England,A
...,...,...,...,...
95,24,4,thee,C
96,25,1,cause,A
97,25,2,fare,B
98,25,3,fair,C


In [24]:
# counts of rhyme words
df['rhyme_word'].value_counts()

rhyme_word
Degree      4
King        3
she         3
fair        2
way         2
           ..
there       1
likewise    1
week        1
Majesty     1
more        1
Name: count, Length: 88, dtype: int64